In [1]:
import os
from sys import platform

import torchaudio

from tortoise.api import TextToSpeech
from tortoise.utils.audio import load_voice


ROOT = os.getcwd()

modelDir = os.path.join(ROOT, "experiments", "Test1", "models")
outputDir = os.path.join(ROOT, "output")

os.makedirs(os.path.join(ROOT, "output"), exist_ok=True)

outputFiles = [file for file in os.listdir(outputDir) if ".DS_Store" not in file]
modelFiles = sorted(
    [
        (int(file.replace("_gpt.pth", "")), os.path.join(modelDir, file))
        for file in (os.listdir(modelDir))
        if ".DS_Store" not in file
    ]
)

models = [model for model, filepath in modelFiles]

newModels = sorted(
    [
        (model, filepath)
        for model, filepath in modelFiles
        if f"{model}.wav" not in outputFiles
    ],
    reverse=True,
)

In [43]:
for model, model_path in newModels:
    autoregressive_batch_size = 8 if platform == "darwin" else None
    tts = TextToSpeech(
        model_path=model_path,
        use_deepspeed=True,
        kv_cache=True,
        autoregressive_batch_size=autoregressive_batch_size,
    )

    text = "You know, liberalism used to be something my dad would get into fights over in bars."

    preset = "fast"

    voice = "john"

    voice_samples, conditioning_latents = load_voice(voice)
    gen = tts.tts_with_preset(
        text,
        voice_samples=voice_samples,
        conditioning_latents=conditioning_latents,
        preset=preset,
    )
    torchaudio.save(f"output/{model}.wav", gen.squeeze(0).cpu(), 24000)
    print(f"{model} saved")
    del tts

In [44]:
import nemo.collections.asr as nemo_asr
import logging

logging.getLogger("nemo_logger").setLevel(logging.ERROR)

speaker_model = nemo_asr.models.EncDecSpeakerLabelModel.from_pretrained(
    "nvidia/speakerverification_en_titanet_large"
)

In [ ]:
import os

ROOT = os.getcwd()

reference = os.path.join(ROOT, "john.wav")

outputDir = os.path.join(ROOT, "output")

files = [
    (file.replace(".wav", ""), os.path.join(outputDir, file))
    for file in os.listdir(outputDir)
    if ".DS_Store" not in file
]

len(files)

In [ ]:
import librosa


def isReference(reference, sample):
    return speaker_model.verify_speakers(reference, sample)


def getDuration(filename):
    duration = librosa.get_duration(path=filename)
    return duration


results = [
    (model, file)
    for model, file in files
    if getDuration(file) < 6
    if isReference(reference, file)
]

len(results)

In [49]:
import os
import shutil

selectedDir = os.path.join(ROOT, "selected")

os.makedirs(selectedDir, exist_ok=True)

for model, filepath in results:
    outpath = os.path.join(selectedDir, f"{model}.wav")
    shutil.copyfile(filepath, outpath)